In [31]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from pathlib import Path
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [32]:
spx_data = pd.read_csv('/Users/tianjiao/Library/CloudStorage/OneDrive-UniversityofOtago/Essay3/PhD_essay3_data/spx_close_return.csv')
spx_data = (
            spx_data
            .drop(['secid', 'return'], axis=1)
            .assign(date = lambda df: pd.to_datetime(df['date']))
            )
spx_data

def ret(df, day):
    df[f'ret_{day}d'] = (df['close'].shift(-day) - df['close']) / df['close'] * 100

for day in [1,5, 7, 10, 15, 22, 44, 66, 132, 190, 252]:
    ret(spx_data, day)

spx_data.drop(columns=['close'], inplace=True)
spx_data

,date,ret_1d,ret_5d,ret_7d,ret_10d,ret_15d,ret_22d,ret_44d,ret_66d,ret_132d,ret_190d,ret_252d
0,1996-01-02,0.095049,-1.817215,-2.906256,-1.979927,-1.279139,2.856314,5.648188,5.659465,5.691686,11.011229,21.445717
1,1996-01-03,-0.582631,-3.676045,-3.140089,-2.406168,-0.218889,2.336960,4.937874,3.688920,3.919076,11.699285,19.220370
2,1996-01-04,-0.160272,-2.429982,-2.894609,-1.531488,-0.108467,3.841671,5.819977,3.964708,4.612271,12.154768,19.315202
3,1996-01-05,0.283764,-2.416046,-1.340987,-0.791296,0.796160,4.802906,2.722511,2.722511,2.122554,13.742278,21.293639
4,1996-01-08,-1.456844,-3.013938,-1.954856,-0.818161,0.931346,5.088445,3.486078,2.056722,1.602367,13.724412,20.888982
...,...,...,...,...,...,...,...,...,...,...,...,...
7460,2025-08-25,0.413398,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7461,2025-08-26,0.239099,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7462,2025-08-27,0.315673,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7463,2025-08-28,-0.639817,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [33]:
r_1990_2023 = (
    pd.read_csv('/Users/tianjiao/Library/CloudStorage/OneDrive-UniversityofOtago/Essay3/PhD_essay3_data/par-yield-curve-rates-1990-2023.csv')
    .rename(columns={'Date': 'date', '1 mo': '1mo', '2 mo': '2mo', '3 mo': '3mo', '4 mo': '4mo', '6 mo': '6mo', '1 yr': '1yr', '2 yr': '2yr', '3 yr': '3yr', '5 yr': '5yr', '7 yr': '7yr', '10 yr': '10yr', '20 yr': '20yr', '30 yr': '30yr'})
    .drop(columns=['2yr', '3yr', '5yr', '7yr', '10yr', '20yr', '30yr'])
)
r_2024 = (
    pd.read_csv('/Users/tianjiao/Library/CloudStorage/OneDrive-UniversityofOtago/Essay3/PhD_essay3_data/daily-treasury-rates-2024.csv')
    .rename(columns={'Date': 'date', '1 Mo': '1mo', '2 Mo': '2mo', '3 Mo': '3mo', '4 Mo': '4mo', '6 Mo': '6mo', '1 Yr': '1yr', '2 Yr': '2yr', '3 Yr': '3yr', '5 Yr': '5yr', '7 Yr': '7yr', '10 Yr': '10yr', '20 Yr': '20yr', '30 Yr': '30yr'})
    .drop(columns=['2yr', '3yr', '5yr', '7yr', '10yr', '20yr', '30yr'])
)
r_2025 = (
    pd.read_csv('/Users/tianjiao/Library/CloudStorage/OneDrive-UniversityofOtago/Essay3/PhD_essay3_data/daily-treasury-rates-2025.csv')
    .rename(columns={'Date': 'date', '1 Mo': '1mo', '2 Mo': '2mo', '3 Mo': '3mo', '4 Mo': '4mo', '6 Mo': '6mo', '1 Yr': '1yr', '2 Yr': '2yr', '3 Yr': '3yr', '5 Yr': '5yr', '7 Yr': '7yr', '10 Yr': '10yr', '20 Yr': '20yr', '30 Yr': '30yr'})
    .drop(columns=['1.5 Month'])
    .drop(columns=['2yr', '3yr', '5yr', '7yr', '10yr', '20yr', '30yr'])
    )
r_df = (
    pd.concat([r_1990_2023, r_2024, r_2025], axis=0, ignore_index=True)
    .assign(date = lambda df: pd.to_datetime(df['date']))
    .sort_values('date')
    .reset_index(drop=True)
)
r_df

,date,1mo,2mo,3mo,4mo,6mo,1yr
0,1990-01-02,NaN,NaN,7.83,NaN,7.89,7.81
1,1990-01-03,NaN,NaN,7.89,NaN,7.94,7.85
2,1990-01-04,NaN,NaN,7.84,NaN,7.90,7.82
3,1990-01-05,NaN,NaN,7.79,NaN,7.85,7.79
4,1990-01-08,NaN,NaN,7.79,NaN,7.88,7.81
...,...,...,...,...,...,...,...
9001,2025-12-24,3.72,3.74,3.69,3.67,3.59,3.50
9002,2025-12-26,3.70,3.72,3.64,3.66,3.58,3.49
9003,2025-12-29,3.69,3.70,3.68,3.66,3.59,3.48
9004,2025-12-30,3.65,3.65,3.65,3.63,3.59,3.47


In [34]:
excess_ret_df = (
    spx_data.merge(r_df, on='date', how='inner')
    .assign(ret_1d = lambda df: df['ret_1d'] - df['3mo'])
    .assign(ret_1w = lambda df: df['ret_5d'] - df['3mo'])
    .assign(ret_9d = lambda df: df['ret_7d'] - df['3mo'])
    .assign(ret_2w = lambda df: df['ret_10d'] - df['3mo'])
    .assign(ret_3w = lambda df: df['ret_15d'] - df['3mo'])
    .assign(ret_1m = lambda df: df['ret_22d'] - df['3mo'])
    .assign(ret_2m = lambda df: df['ret_44d'] - df['3mo'])
    .assign(ret_3m = lambda df: df['ret_66d'] - df['3mo'])
    .assign(ret_6m = lambda df: df['ret_132d'] - df['3mo'])
    .assign(ret_9m = lambda df: df['ret_190d'] - df['3mo'])
    .assign(ret_1y = lambda df: df['ret_252d'] - df['3mo'])
    .filter(items=['date', 'ret_1d', 'ret_1w', 'ret_9d', 'ret_2w', 'ret_3w', 'ret_1m', 'ret_2m', 'ret_3m', 'ret_6m', 'ret_9m', 'ret_1y'])
    .sort_values('date')    
    .reset_index(drop=True)
)
excess_ret_df

,date,ret_1d,ret_1w,ret_9d,ret_2w,ret_3w,ret_1m,ret_2m,ret_3m,ret_6m,ret_9m,ret_1y
0,1996-01-02,-5.104951,-7.017215,-8.106256,-7.179927,-6.479139,-2.343686,0.448188,0.459465,0.491686,5.811229,16.245717
1,1996-01-03,-5.782631,-8.876045,-8.340089,-7.606168,-5.418889,-2.863040,-0.262126,-1.511080,-1.280924,6.499285,14.020370
2,1996-01-04,-5.350272,-7.619982,-8.084609,-6.721488,-5.298467,-1.348329,0.629977,-1.225292,-0.577729,6.964768,14.125202
3,1996-01-05,-4.906236,-7.606046,-6.530987,-5.981296,-4.393840,-0.387094,-2.467489,-2.467489,-3.067446,8.552278,16.103639
4,1996-01-08,-6.636844,-8.193938,-7.134856,-5.998161,-4.248654,-0.091555,-1.693922,-3.123278,-3.577633,8.544412,15.708982
...,...,...,...,...,...,...,...,...,...,...,...,...
7405,2025-08-25,-3.876602,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7406,2025-08-26,-4.040901,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7407,2025-08-27,-3.944327,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7408,2025-08-28,-4.899817,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [35]:
all_vix_df = (
    pd.read_csv('/Users/tianjiao/Library/CloudStorage/OneDrive-UniversityofOtago/Essay2/PhD_essay2_empirical/all_vix.csv')
    .assign(date = lambda df: pd.to_datetime(df['Date']))
    .drop(columns=['Date'])
    .sort_values('date')
    .reset_index(drop=True)
)
vix_eret_df = (
    all_vix_df.merge(excess_ret_df, on='date', how='inner')
    .filter(items=['date', 'ret_1d', 'VIX1W', 'ret_1w', 'CBOEVIX9D', 'ret_9d', 'VIX2W', 'ret_2w', 'VIX3W', 'ret_3w', 'CBOEVIX', 'ret_1m', 'VIX2M', 'ret_2m', 'CBOEVIX3M', 'ret_3m', 'CBOEVIX6M', 'ret_6m', 'VIX9M', 'ret_9m', 'CBOEVIX1Y', 'ret_1y'])
    .dropna()
    .sort_values('date')
    .reset_index(drop=True)
    # .assign(**{f'{vix}_2': lambda df: df[vix] ** 2 for vix in ['CBOEVIX1D', 'VIX1W', 'CBOEVIX9D', 'VIX2W', 'VIX3W', 'CBOEVIX', 'VIX2M', 'CBOEVIX3M', 'CBOEVIX6M']})
    # .assign(**{f'{vix}_delta': lambda df: df[f'{vix}_2'] - df[f'{vix}_2'].shift(1) for vix in ['CBOEVIX1D', 'VIX1W', 'CBOEVIX9D', 'VIX2W', 'VIX3W', 'CBOEVIX', 'VIX2M', 'CBOEVIX3M', 'CBOEVIX6M']})
)
vix_eret_df

,date,ret_1d,VIX1W,ret_1w,CBOEVIX9D,ret_9d,VIX2W,ret_2w,VIX3W,ret_3w,...,VIX2M,ret_2m,CBOEVIX3M,ret_3m,CBOEVIX6M,ret_6m,VIX9M,ret_9m,CBOEVIX1Y,ret_1y
0,2011-01-04,0.360709,18.843772,0.196955,16.06,0.927548,19.850429,0.782689,20.174822,1.940775,...,18.966022,3.782217,20.61,4.423848,23.19,2.904402,27.353019,-10.073081,25.80,0.418967
1,2011-01-05,-0.352289,19.319286,0.596354,15.57,1.166637,19.771523,0.149841,19.919988,1.660150,...,18.487206,1.313124,20.05,3.612272,22.78,2.960520,27.106234,-8.881461,25.62,0.212510
2,2011-01-06,-0.334480,12.361377,0.627957,15.71,1.511891,16.632511,0.595771,18.278048,0.045470,...,18.874071,2.238821,20.35,3.014423,22.87,2.329884,27.061544,-9.443873,25.61,0.160869
3,2011-01-07,-0.277633,12.713897,1.569792,15.01,0.679505,17.228448,1.381038,18.489948,1.009823,...,18.698094,1.817530,20.29,3.234754,22.92,4.203689,27.023363,-6.165167,25.57,0.583555
4,2011-01-10,0.222514,18.297197,1.840156,15.81,0.677722,19.501473,1.537734,19.886699,2.830114,...,18.952617,0.804519,20.48,3.375891,22.93,4.267405,26.961641,-5.994458,25.81,1.608614
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3347,2024-08-21,-6.153281,13.973502,-5.770065,15.65,-4.769861,16.037202,-7.349364,18.216874,-5.706374,...,26.621693,-2.118660,18.58,0.939952,19.54,-2.461491,108.199665,0.089547,21.44,9.301321
3348,2024-08-22,-4.131658,17.274003,-4.897279,16.55,-6.028747,17.581916,-8.192053,20.010985,-4.285859,...,28.793533,-0.985700,19.36,2.200828,20.09,-0.397437,114.790868,0.426885,21.65,10.791762
3349,2024-08-23,-5.565372,13.751930,-5.005263,13.69,-7.282794,15.685717,-8.152774,17.426013,-5.276976,...,25.409928,-2.170639,18.19,1.618621,19.35,-3.405862,94.771806,-0.324016,21.24,9.778369
3350,2024-08-26,-5.080480,13.557324,-6.805115,14.87,-7.259463,15.524490,-7.399933,17.436245,-4.924164,...,27.319815,-1.560351,18.25,1.559197,19.36,-2.509639,95.754745,0.009393,20.27,10.516546


In [36]:
def adj_r2(df, d, vix):
    df = df.copy()
    df[f'{vix}_2'] = df[vix] ** 2
    df[f'{vix}_delta'] = df[f'{vix}_2'] - df[f'{vix}_2'].shift(1)
    # df[f'ret_{d}_lag1'] = df[f'ret_{d}'].shift(1)
    df.dropna(inplace=True)
    x = df[[f'{vix}_delta']]
    y = df[f'{vix}_2']
    model = LinearRegression()
    model.fit(x, y)
    df[f'{vix}_resid'] = y - model.predict(x)

    x_r = sm.add_constant(df[[f'{vix}_delta', f'{vix}_resid']])
    y_r = df[f'ret_{d}']
    model_r = sm.OLS(y_r, x_r).fit()
    
    print(model_r.summary())
    
    return model_r.rsquared_adj * 100

print(adj_r2(vix_eret_df, '1d', 'VIX1W'))

ds=['1d', '1w', '9d', '2w', '3w', '1m', '2m', '3m', '6m']
vix = ['VIX1W', 'CBOEVIX9D', 'VIX2W', 'VIX3W', 'CBOEVIX', 'VIX2M', 'CBOEVIX3M', 'CBOEVIX6M']

# numeric table — for analysis
table_adj_r2_num = pd.DataFrame(columns=[f'ret{d}' for d in ds], index=vix, dtype=float)

# formatted table — for latex
table_adj_r2_fmt = pd.DataFrame(columns=[f'ret{d}' for d in ds], index=vix)

for d in ds:
    for v in vix:
        table_adj_r2_num.loc[v, f'ret{d}'] = adj_r2(vix_eret_df, d, v)

# format after all values computed
for d in ds:
    max_v = table_adj_r2_num[f'ret{d}'].astype(float).idxmax()
    for v in vix:
        val = f"{table_adj_r2_num.loc[v, f'ret{d}']:.2f}"
        if v == max_v:
            val = f"\\textbf{{{val}}}"
        table_adj_r2_fmt.loc[v, f'ret{d}'] = val

# Export to LaTeX
table_adj_r2_fmt.columns = ['R1D', 'R1W', 'R9D', 'R2W', 'R3W', 'R1M', 'R2M', 'R3M', 'R6M']
file_path = Path(f"/Users/tianjiao/Library/CloudStorage/OneDrive-UniversityofOtago/Essay3/PhD_essay3_writing/tables/adj_r2_table.tex")
table_adj_r2_fmt.to_latex(
    file_path,
    column_format="l" + "c"*len(ds),
    escape=False
)

table_adj_r2_fmt

                            OLS Regression Results                            
Dep. Variable:                 ret_1d   R-squared:                       0.021
Model:                            OLS   Adj. R-squared:                  0.020
Method:                 Least Squares   F-statistic:                     35.47
Date:                Tue, 30 Jun 2026   Prob (F-statistic):           5.69e-16
Time:                        14:08:06   Log-Likelihood:                -7188.8
No. Observations:                3351   AIC:                         1.438e+04
Df Residuals:                    3348   BIC:                         1.440e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const          -1.2113      0.036    -33.902      

,R1D,R1W,R9D,R2W,R3W,R1M,R2M,R3M,R6M
VIX1W,2.02,0.97,1.43,2.24,3.93,6.54,9.00,10.91,11.75
CBOEVIX9D,2.24,1.06,1.52,2.36,4.31,6.83,9.55,11.57,12.16
VIX2W,1.85,0.92,1.48,2.27,4.16,6.82,9.93,12.28,13.10
VIX3W,1.38,0.81,1.37,2.01,3.88,6.52,10.21,13.05,13.76
CBOEVIX,2.36,1.98,2.71,3.60,5.65,8.38,11.72,\textbf{13.65},\textbf{14.11}
VIX2M,\textbf{5.67},0.78,0.20,-0.03,0.46,1.80,4.87,7.55,10.60
CBOEVIX3M,2.48,3.39,\textbf{4.01},\textbf{4.65},\textbf{6.85},\textbf{8.92},\textbf{11.99},13.47,13.22
CBOEVIX6M,2.71,\textbf{3.46},3.97,4.48,6.12,7.84,10.36,11.40,10.40


In [37]:
from pathlib import Path
import pandas as pd
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

def run_pc_reg_table(excess_ret_df, vix_eret_df, vix_cols, ds, n_pcs, col_names, file_path, loadings=False):
    """
    excess_ret_df  : DataFrame with date and ret_{d} columns
    vix_eret_df    : DataFrame with date and VIX columns
    vix_cols       : list of VIX column names e.g. ['VIX1W', 'CBOEVIX9D', ...]
    ds             : list of return horizons e.g. ['1d', '1w', ...]
    n_pcs          : number of PCs to use e.g. 2 or 3
    file_path      : Path to save latex table
    loadings       : whether to return loadings DataFrame
    """
    # 1. scale
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(vix_eret_df[vix_cols])

    # 2. fit PCA
    pca = PCA()
    X_pca = pca.fit_transform(X_scaled)

    # 3. PC scores with date
    pc_df = pd.DataFrame(X_pca, columns=[f'PC{i+1}' for i in range(X_pca.shape[1])], index=vix_eret_df.index)
    pc_df.insert(0, 'date', vix_eret_df['date'].values)

    # 4. variance explained
    variance_df = pd.DataFrame({
        'variance_explained': pca.explained_variance_ratio_,
        'cumulative': pca.explained_variance_ratio_.cumsum()
    }, index=[f'PC{i+1}' for i in range(len(pca.explained_variance_ratio_))])

    # 5. loadings
    loadings_df = pd.DataFrame(pca.components_, columns=vix_cols, index=[f'PC{i+1}' for i in range(len(vix_cols))])

    if loadings:
        print(variance_df)
        print(loadings_df)

    # 6. merge with excess returns
    pcs = [f'PC{i+1}' for i in range(n_pcs)]
    df = (
        excess_ret_df.merge(pc_df, on='date', how='inner')
        .filter(items=['date'] + [f'ret_{d}' for d in ds] + pcs)
        .dropna()
        .sort_values('date')
        .reset_index(drop=True)
    )

    # 7. run all regressions
    models = {}
    for d in ds:
        data = df.dropna().copy()
        x_r = sm.add_constant(data[pcs])
        y_r = data[f'ret_{d}']
        models[f'ret_{d}'] = sm.OLS(y_r, x_r).fit()
        print(models[f'ret_{d}'].summary())

    # 8. format coefficient with stars
    def format_coef(model, var):
        coef = model.params[var]
        se   = model.bse[var]
        pval = model.pvalues[var]
        if pval < 0.01:
            stars = '^{***}'
        elif pval < 0.05:
            stars = '^{**}'
        elif pval < 0.1:
            stars = '^{*}'
        else:
            stars = ''
        return f"${coef:.3f}{stars}$", f"$({se:.3f})$"

    # 9. build rows dynamically from pcs
    rows = []
    for pc in pcs:
        rows += [pc, f'{pc}_se']
    rows += ['const', 'const_se', 'N', 'Adj. $R^2$']

    table_df = pd.DataFrame(index=rows, columns=[f'ret_{d}' for d in ds])

    for d, model in models.items():
        for var in pcs + ['const']:
            coef_str, se_str = format_coef(model, var)
            table_df.loc[var, d]         = coef_str
            table_df.loc[f'{var}_se', d] = se_str
        table_df.loc['N', d]          = f"${int(model.nobs)}$"
        table_df.loc['Adj. $R^2$', d] = f"${model.rsquared_adj*100:.2f}$"

    # 10. rename se rows to blank
    rename_map = {f'{var}_se': ' ' * (i+1) for i, var in enumerate(pcs + ['const'])}
    table_df = table_df.rename(index=rename_map)

    # 11. export
    table_df.columns = col_names[:len(ds)]
    table_df.to_latex(file_path, escape=False, column_format="l" + "c" * len(ds))

    return table_df, variance_df, loadings_df


table_df, variance_df, loadings_df = run_pc_reg_table(
    excess_ret_df  = excess_ret_df,
    vix_eret_df    = vix_eret_df,
    vix_cols       = ['VIX1W', 'CBOEVIX9D', 'VIX2W', 'VIX3W', 'CBOEVIX', 'VIX2M', 'CBOEVIX3M', 'CBOEVIX6M'],
    col_names     = ['R1D', 'R1W', 'R9D', 'R2W', 'R3W', 'R1M', 'R2M', 'R3M', 'R6M'],
    ds             = ['1d', '1w', '9d', '2w', '3w', '1m', '2m', '3m', '6m'],
    n_pcs          = 3,
    file_path      = Path("/Users/tianjiao/Library/CloudStorage/OneDrive-UniversityofOtago/Essay3/PhD_essay3_writing/tables/pc_table.tex"),
    loadings       = True
)
table_df

     variance_explained  cumulative
PC1            0.926330    0.926330
PC2            0.039869    0.966198
PC3            0.025222    0.991420
PC4            0.003531    0.994952
PC5            0.002890    0.997841
PC6            0.001055    0.998897
PC7            0.000780    0.999677
PC8            0.000323    1.000000
        VIX1W  CBOEVIX9D     VIX2W     VIX3W   CBOEVIX     VIX2M  CBOEVIX3M  \
PC1  0.351138   0.357642  0.361357  0.363235  0.364710  0.332899   0.356015   
PC2 -0.473565  -0.336099 -0.270684 -0.115749  0.009539  0.243717   0.372727   
PC3 -0.081148  -0.115307  0.001636  0.084681 -0.168567  0.886639  -0.264119   
PC4 -0.234196  -0.498800  0.388582  0.700743 -0.099249 -0.200876  -0.052182   
PC5  0.685639  -0.415426  0.073042 -0.149777 -0.456378  0.030609  -0.066601   
PC6 -0.341768   0.434360  0.511019 -0.163977 -0.502713 -0.015730  -0.226451   
PC7 -0.064027  -0.350443  0.615040 -0.551495  0.307915  0.045803   0.209075   
PC8 -0.000662   0.113911 -0.020072  0.054608

,R1D,R1W,R9D,R2W,R3W,R1M,R2M,R3M,R6M
PC1,$0.047^{***}$,$0.106^{***}$,$0.133^{***}$,$0.182^{***}$,$0.278^{***}$,$0.409^{***}$,$0.641^{***}$,$0.817^{***}$,$1.070^{***}$
,$(0.008)$,$(0.015)$,$(0.017)$,$(0.019)$,$(0.023)$,$(0.027)$,$(0.034)$,$(0.039)$,$(0.051)$
PC2,$-0.156^{***}$,$-0.001$,$0.078$,$0.045$,$-0.016$,$-0.251^{*}$,$-0.284^{*}$,$-0.594^{***}$,$-1.217^{***}$
,$(0.037)$,$(0.070)$,$(0.080)$,$(0.094)$,$(0.112)$,$(0.130)$,$(0.166)$,$(0.188)$,$(0.245)$
PC3,$-3.788^{***}$,$-3.732^{***}$,$-3.692^{***}$,$-3.643^{***}$,$-3.612^{***}$,$-3.550^{***}$,$-3.135^{***}$,$-2.392^{***}$,$-0.506$
,$(0.046)$,$(0.088)$,$(0.100)$,$(0.118)$,$(0.141)$,$(0.164)$,$(0.209)$,$(0.237)$,$(0.308)$
const,$-1.211^{***}$,$-1.017^{***}$,$-0.927^{***}$,$-0.790^{***}$,$-0.543^{***}$,$-0.193^{***}$,$0.849^{***}$,$1.887^{***}$,$5.100^{***}$
,$(0.021)$,$(0.040)$,$(0.045)$,$(0.053)$,$(0.063)$,$(0.073)$,$(0.094)$,$(0.106)$,$(0.138)$
N,$3352$,$3352$,$3352$,$3352$,$3352$,$3352$,$3352$,$3352$,$3352$
Adj. $R^2$,$66.84$,$35.57$,$29.70$,$23.68$,$19.15$,$17.29$,$14.60$,$13.99$,$12.24$


In [50]:
vix_df_wang2019 = (
    pd.read_csv('/Users/tianjiao/Library/CloudStorage/OneDrive-UniversityofOtago/Essay2/PhD_essay2_empirical/all_vix.csv')
    .assign(date = lambda df: pd.to_datetime(df['Date']))
    .drop(columns=['Date'])
    .sort_values('date')
    .reset_index(drop=True)
)
vix_eret_df_wang2019 = (
    vix_df_wang2019.merge(excess_ret_df, on='date', how='inner')
    .filter(items=['date', 'CBOEVIX', 'ret_1m', 'VIX2M', 'ret_2m', 'VIX3M', 'ret_3m', 'VIX6M', 'ret_6m', 'VIX9M', 'ret_9m', 'VIX1Y', 'ret_1y'])
    .dropna()
    .sort_values('date')
    .loc[lambda df: (df['date'] >= '1996-01-04') & (df['date'] <= '2016-04-29')]
    .reset_index(drop=True)
    # .assign(**{f'{vix}_2': lambda df: df[vix] ** 2 for vix in ['CBOEVIX1D', 'VIX1W', 'CBOEVIX9D', 'VIX2W', 'VIX3W', 'CBOEVIX', 'VIX2M', 'CBOEVIX3M', 'CBOEVIX6M']})
    # .assign(**{f'{vix}_delta': lambda df: df[f'{vix}_2'] - df[f'{vix}_2'].shift(1) for vix in ['CBOEVIX1D', 'VIX1W', 'CBOEVIX9D', 'VIX2W', 'VIX3W', 'CBOEVIX', 'VIX2M', 'CBOEVIX3M', 'CBOEVIX6M']})
)
vix_eret_df_wang2019


table_df_wang2019, variance_df_wang2019, loadings_df_wang2019 = run_pc_reg_table(
    excess_ret_df  = excess_ret_df,
    vix_eret_df    = vix_eret_df_wang2019,
    vix_cols       = ['CBOEVIX', 'VIX2M', 'VIX3M', 'VIX6M', 'VIX9M', 'VIX1Y'],
    col_names     = ['R1M', 'R2M', 'R3M', 'R6M', 'R9M', 'R1Y'],
    ds             = ['1m', '2m', '3m', '6m', '9m', '1y'],
    n_pcs          = 2,
    file_path      = Path("/Users/tianjiao/Library/CloudStorage/OneDrive-UniversityofOtago/Essay3/PhD_essay3_writing/tables/longterm_pc_table.tex"),
    loadings       = True
)
table_df_wang2019

     variance_explained  cumulative
PC1            0.673635    0.673635
PC2            0.278142    0.951776
PC3            0.022661    0.974438
PC4            0.011218    0.985656
PC5            0.010648    0.996303
PC6            0.003697    1.000000
      CBOEVIX     VIX2M     VIX3M     VIX6M     VIX9M     VIX1Y
PC1  0.318225  0.400189  0.471383  0.447720  0.412436  0.381867
PC2  0.582126  0.449493  0.179682 -0.250834 -0.401707 -0.450016
PC3 -0.181240 -0.084613  0.074459  0.722948 -0.040717 -0.655849
PC4 -0.429255 -0.109591  0.798084 -0.158142 -0.369595  0.071991
PC5 -0.129816  0.019328  0.225008 -0.434328  0.726190 -0.464922
PC6 -0.570879  0.786296 -0.228914 -0.018046 -0.054017  0.013789
                            OLS Regression Results                            
Dep. Variable:                 ret_1m   R-squared:                       0.102
Model:                            OLS   Adj. R-squared:                  0.102
Method:                 Least Squares   F-statistic:           

,R1M,R2M,R3M,R6M,R9M,R1Y
PC1,$-0.561^{***}$,$-0.528^{***}$,$-0.566^{***}$,$-0.427^{***}$,$-0.494^{***}$,$-0.802^{***}$
,$(0.042)$,$(0.058)$,$(0.070)$,$(0.102)$,$(0.123)$,$(0.139)$
PC2,$0.994^{***}$,$1.210^{***}$,$1.234^{***}$,$2.323^{***}$,$2.861^{***}$,$3.615^{***}$
,$(0.065)$,$(0.090)$,$(0.109)$,$(0.159)$,$(0.191)$,$(0.216)$
const,$-0.879^{***}$,$-0.412^{***}$,$0.120$,$1.902^{***}$,$3.627^{***}$,$5.627^{***}$
,$(0.084)$,$(0.117)$,$(0.140)$,$(0.205)$,$(0.247)$,$(0.279)$
N,$3675$,$3675$,$3675$,$3675$,$3675$,$3675$
Adj. $R^2$,$10.17$,$6.61$,$4.98$,$5.86$,$6.08$,$7.81$


In [60]:
vix_df_wang2019 = (
    vix_df_wang2019
    .filter(items=['date', 'CBOEVIX', 'VIX2M', 'VIX3M', 'VIX6M', 'VIX9M', 'VIX1Y'])
    .dropna()
)
corr_table = vix_df_wang2019.drop(columns='date').corr()
corr_table

,CBOEVIX,VIX2M,VIX3M,VIX6M,VIX9M,VIX1Y
CBOEVIX,1.000000,0.903919,0.702201,0.265392,0.103989,0.020132
VIX2M,0.903919,1.000000,0.904047,0.572859,0.419916,0.306270
VIX3M,0.702201,0.904047,1.000000,0.803571,0.691240,0.569871
VIX6M,0.265392,0.572859,0.803571,1.000000,0.917155,0.800777
VIX9M,0.103989,0.419916,0.691240,0.917155,1.000000,0.878152
VIX1Y,0.020132,0.306270,0.569871,0.800777,0.878152,1.000000
